# 07 — Evaluación del clasificador acústico sobre el conjunto de prueba

Este notebook compara las predicciones generadas por `06_Workflow_predecir.ipynb`
con anotaciones manuales de Raven Pro. Está diseñado para evaluar un conjunto de
prueba **independiente** y producir resultados reproducibles a nivel de ventana,
evento y archivo de audio.

El ejemplo usa la salida `agile_classifier_v8_pred.csv`, la etiqueta `PRTR1` y un
umbral logit de `0.01`, con lo que también permite comprobar los resultados del
script original `AMISTOSA_evaluacion_modelo_campana.R`.

## Decisiones metodológicas

1. **Umbral bloqueado:** el resultado principal usa `DECISION_THRESHOLD`, que debe
   definirse antes de observar el test (idealmente con train/validation). El mejor
   F1 calculado sobre test se presenta solo como diagnóstico exploratorio.
2. **Asignación temporal:** por defecto, cada anotación se asigna a la ventana que
   contiene su punto medio. Así una llamada corta cuenta una sola vez y se evitan
   dobles asignaciones en los límites. También se comparan las reglas `start` y
   `any_overlap`.
3. **Ventanas semiabiertas:** se usa `[window_start, window_end)`, por lo que un
   evento situado exactamente en un límite pertenece a una sola ventana.
4. **Desbalance de clases:** además de accuracy se reportan precision, recall,
   specificity, F1, balanced accuracy, MCC, ROC AUC, average precision y falsos
   positivos por hora.
5. **Incertidumbre:** los intervalos de confianza se calculan mediante bootstrap
   agrupado por archivo, preservando la dependencia entre ventanas del mismo audio.

In [ ]:
# @title Imports

from pathlib import Path
import json
import math
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    matthews_corrcoef,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [ ]:
# @title Configuración — editar esta celda

DATA_DIR = Path('/mnt/d/Test_data_campana/test_data/Performance 2026' )
PREDICTIONS_CSV = DATA_DIR / "agile_classifier_v8_pred.csv"
ANNOTATIONS_CSV = DATA_DIR / "anotaciones.csv"
DURATIONS_CSV = DATA_DIR / "duracion.csv"
OUTPUT_DIR = DATA_DIR / "07_evaluacion_modelo"

# La etiqueta producida por el clasificador y las etiquetas Raven que cuentan
# como presencia del objetivo. Para una evaluación a nivel de especie podrían
# incluirse, por ejemplo, ("PRTR1", "PRTR2", "PRTR3").
PREDICTION_LABEL = "PRTR1"
TARGET_ANNOTATION_LABELS = ("PRTR1",)

# None incluye todas las calidades. Si se usa ("A", "B"), las ventanas que
# contengan anotaciones objetivo C se excluyen, en vez de tratarlas como negativas.
ACCEPTED_QUALITIES = None
QUALITY_COLUMN = "Calidad"

# Etiquetas Raven que deben crear zonas ignoradas, no negativos verdaderos.
# Ejemplo posible: ("PRTR2", "PRTR3") si son vocalizaciones ambiguas para PRTR1.
IGNORED_ANNOTATION_LABELS = ("PRTR2", "PRTR3")

SCORE_COLUMN = "logits"
SCORE_TYPE = "logit"        # "logit" o "probability"
DECISION_THRESHOLD = 0.01    # umbral bloqueado; 0 logit equivale a p=0.5

TRUTH_ASSIGNMENT = "midpoint"  # "midpoint", "start" o "any_overlap"
MIN_OVERLAP_SECONDS = 0.0       # solo se usa con any_overlap
EVENT_TOLERANCE_SECONDS = 0.0

BOOTSTRAP_REPS = 1_000
RANDOM_SEED = 42
SAVE_OUTPUTS = True

assert SCORE_TYPE in {"logit", "probability"}
assert TRUTH_ASSIGNMENT in {"midpoint", "start", "any_overlap"}

In [ ]:
# @title Funciones de lectura y normalización

def read_csv_flexible(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"No existe: {path.resolve()}")
    df = pd.read_csv(path)
    if df.shape[1] == 1:
        df = pd.read_csv(path, sep=None, engine="python")
    df.columns = [str(c).strip().lstrip("\ufeff") for c in df.columns]
    return df


def to_numeric_flexible(series, column_name):
    cleaned = series.astype(str).str.strip().str.replace(",", ".", regex=False)
    numeric = pd.to_numeric(cleaned, errors="coerce")
    bad = numeric.isna() & series.notna()
    if bad.any():
        examples = series[bad].astype(str).head(5).tolist()
        raise ValueError(f"Valores no numéricos en {column_name}: {examples}")
    return numeric


def basename_any(value):
    return re.split(r"[\\/]", str(value).strip())[-1]


def require_columns(df, required, table_name):
    missing = sorted(set(required) - set(df.columns))
    if missing:
        raise ValueError(f"Faltan columnas en {table_name}: {missing}")


def score_to_probability(values):
    values = np.asarray(values, dtype=float)
    return expit(values) if SCORE_TYPE == "logit" else values


def threshold_to_probability(value):
    return float(expit(value)) if SCORE_TYPE == "logit" else float(value)

In [ ]:
# @title Cargar, filtrar y validar los tres insumos

ann = read_csv_flexible(ANNOTATIONS_CSV)
pred = read_csv_flexible(PREDICTIONS_CSV)
durations = read_csv_flexible(DURATIONS_CSV)

require_columns(
    pred,
    ["filename", "window_start", "window_end", SCORE_COLUMN],
    "predicciones",
)
require_columns(
    ann,
    ["filename", "label", "File Offset (s)", "duration"],
    "anotaciones",
)
require_columns(durations, ["filename", "duration"], "duraciones")

for df in (ann, pred, durations):
    df["filename"] = df["filename"].map(basename_any)

pred["window_start"] = to_numeric_flexible(pred["window_start"], "window_start")
pred["window_end"] = to_numeric_flexible(pred["window_end"], "window_end")
pred[SCORE_COLUMN] = to_numeric_flexible(pred[SCORE_COLUMN], SCORE_COLUMN)
ann["File Offset (s)"] = to_numeric_flexible(ann["File Offset (s)"], "File Offset (s)")
ann["duration"] = to_numeric_flexible(ann["duration"], "annotation duration")
durations["duration"] = to_numeric_flexible(durations["duration"], "audio duration")

if "label" in pred.columns and PREDICTION_LABEL is not None:
    available_prediction_labels = sorted(pred["label"].dropna().astype(str).unique())
    pred = pred[pred["label"].astype(str).eq(PREDICTION_LABEL)].copy()
    if pred.empty:
        raise ValueError(
            f"No hay predicciones para {PREDICTION_LABEL!r}. "
            f"Etiquetas disponibles: {available_prediction_labels}"
        )

pred = pred.sort_values(["filename", "window_start", "window_end"]).reset_index(drop=True)
ann = ann.reset_index(drop=True)
ann["annotation_id"] = np.arange(1, len(ann) + 1)

if pred.duplicated(["filename", "window_start", "window_end"]).any():
    raise ValueError("Hay ventanas de predicción duplicadas después de filtrar la etiqueta.")
if durations["filename"].duplicated().any():
    raise ValueError("Hay filenames duplicados en duracion.csv.")
if (pred["window_end"] <= pred["window_start"]).any():
    raise ValueError("Hay ventanas con duración nula o negativa.")
if (ann["duration"] <= 0).any() or (ann["File Offset (s)"] < 0).any():
    raise ValueError("Hay anotaciones con tiempos inválidos.")

files_pred = set(pred["filename"])
files_duration = set(durations["filename"])
if files_pred != files_duration:
    raise ValueError(
        "Predicciones y duraciones no contienen exactamente los mismos archivos. "
        f"Solo predicciones: {len(files_pred - files_duration)}; "
        f"solo duraciones: {len(files_duration - files_pred)}"
    )

annotations_without_predictions = sorted(set(ann["filename"]) - files_pred)
if annotations_without_predictions:
    raise ValueError(
        "Hay anotaciones sin predicciones. Ejemplos: "
        f"{annotations_without_predictions[:5]}"
    )

ann = ann.merge(
    durations.rename(columns={"duration": "audio_duration"}),
    on="filename",
    how="left",
    validate="many_to_one",
)
ann["annotation_start"] = ann["File Offset (s)"]
ann["annotation_end_raw"] = ann["annotation_start"] + ann["duration"]
ann["annotation_end"] = np.minimum(ann["annotation_end_raw"], ann["audio_duration"])
ann["annotation_was_clipped"] = ann["annotation_end_raw"] > ann["audio_duration"] + 1e-6
ann["annotation_midpoint"] = (ann["annotation_start"] + ann["annotation_end"]) / 2

if (ann["annotation_start"] >= ann["audio_duration"]).any():
    bad = ann.loc[ann["annotation_start"] >= ann["audio_duration"], "annotation_id"].tolist()
    raise ValueError(f"Anotaciones que empiezan fuera del audio: {bad[:10]}")

print(f"Predicciones: {len(pred):,} ventanas en {pred['filename'].nunique():,} archivos")
print(f"Anotaciones Raven: {len(ann):,} eventos en {ann['filename'].nunique():,} archivos")
print(f"Duración total de audios: {durations['duration'].sum()/3600:.3f} h")
print(f"Anotaciones recortadas al final del audio: {ann['annotation_was_clipped'].sum():,}")

In [ ]:
# @title Auditoría de cobertura de las predicciones

pred["window_duration"] = pred["window_end"] - pred["window_start"]
typical_window_seconds = float(pred["window_duration"].median())

coverage = (
    pred.groupby("filename", as_index=False)
    .agg(
        observed_windows=("filename", "size"),
        first_window_start=("window_start", "min"),
        last_window_end=("window_end", "max"),
        modeled_seconds=("window_duration", "sum"),
    )
    .merge(durations, on="filename", how="left", validate="one_to_one")
)
coverage["expected_full_windows"] = np.floor(
    (coverage["duration"] + 1e-9) / typical_window_seconds
).astype(int)
coverage["window_count_ok"] = (
    coverage["observed_windows"] == coverage["expected_full_windows"]
)
coverage["unmodeled_tail_seconds"] = coverage["duration"] - coverage["last_window_end"]

if (coverage["last_window_end"] > coverage["duration"] + 1e-6).any():
    raise ValueError("Hay ventanas que terminan después de la duración del audio.")

print(f"Duración típica de ventana: {typical_window_seconds:g} s")
print(f"Cobertura modelada: {coverage['modeled_seconds'].sum()/3600:.3f} h")
print(
    "Archivos con el número esperado de ventanas completas: "
    f"{coverage['window_count_ok'].sum():,}/{len(coverage):,}"
)
display(coverage["unmodeled_tail_seconds"].describe().to_frame("seconds"))
display(coverage.loc[~coverage["window_count_ok"]].head(10))

## Construcción del ground truth por ventana

`TARGET_ANNOTATION_LABELS` define qué eventos son positivos. Si se filtran
calidades o se indican `IGNORED_ANNOTATION_LABELS`, las ventanas afectadas se
excluyen de las métricas: una anotación incierta nunca se convierte silenciosamente
en un negativo verdadero.

In [ ]:
# @title Funciones de correspondencia temporal

def overlap_matrix(window_start, window_end, event_start, event_end, tolerance=0.0):
    ws = np.asarray(window_start, dtype=float)[None, :]
    we = np.asarray(window_end, dtype=float)[None, :]
    es = (np.asarray(event_start, dtype=float) - tolerance)[:, None]
    ee = (np.asarray(event_end, dtype=float) + tolerance)[:, None]
    return np.minimum(we, ee) - np.maximum(ws, es)


def mark_windows(windows, events, rule="midpoint", min_overlap_seconds=0.0):
    marked = pd.Series(False, index=windows.index, dtype=bool)
    if events.empty:
        return marked

    for filename, event_group in events.groupby("filename", sort=False):
        window_group = windows[windows["filename"].eq(filename)]
        if window_group.empty:
            continue

        ws = window_group["window_start"].to_numpy()
        we = window_group["window_end"].to_numpy()

        if rule == "start":
            point = event_group["annotation_start"].to_numpy()[:, None]
            hit = (point >= ws[None, :]) & (point < we[None, :])
        elif rule == "midpoint":
            point = event_group["annotation_midpoint"].to_numpy()[:, None]
            hit = (point >= ws[None, :]) & (point < we[None, :])
        elif rule == "any_overlap":
            overlap = overlap_matrix(
                ws,
                we,
                event_group["annotation_start"],
                event_group["annotation_end"],
            )
            hit = overlap > min_overlap_seconds
        else:
            raise ValueError(f"Regla temporal desconocida: {rule}")

        marked.loc[window_group.index] = hit.any(axis=0)

    return marked


target_ann = ann[ann["label"].astype(str).isin(TARGET_ANNOTATION_LABELS)].copy()

if ACCEPTED_QUALITIES is None:
    accepted_ann = target_ann.copy()
    quality_excluded_ann = target_ann.iloc[0:0].copy()
else:
    require_columns(ann, [QUALITY_COLUMN], "anotaciones")
    accepted_mask = target_ann[QUALITY_COLUMN].astype(str).isin(ACCEPTED_QUALITIES)
    accepted_ann = target_ann[accepted_mask].copy()
    quality_excluded_ann = target_ann[~accepted_mask].copy()

label_ignored_ann = ann[
    ann["label"].astype(str).isin(IGNORED_ANNOTATION_LABELS)
].copy()
ignored_ann = (
    pd.concat([quality_excluded_ann, label_ignored_ann], ignore_index=True)
    .drop_duplicates("annotation_id")
)

evaluation = pred.copy()
evaluation["truth"] = mark_windows(
    evaluation,
    accepted_ann,
    rule=TRUTH_ASSIGNMENT,
    min_overlap_seconds=MIN_OVERLAP_SECONDS,
)
evaluation["ignored"] = mark_windows(
    evaluation,
    ignored_ann,
    rule="any_overlap",
    min_overlap_seconds=0.0,
)
evaluation["score"] = evaluation[SCORE_COLUMN].astype(float)
evaluation["probability"] = score_to_probability(evaluation["score"])
evaluation["predicted"] = evaluation["score"] >= DECISION_THRESHOLD

eval_scored = evaluation.loc[~evaluation["ignored"]].copy()
eval_scored["error_type"] = np.select(
    [
        eval_scored["truth"] & eval_scored["predicted"],
        ~eval_scored["truth"] & eval_scored["predicted"],
        eval_scored["truth"] & ~eval_scored["predicted"],
    ],
    ["TP", "FP", "FN"],
    default="TN",
)

print(f"Anotaciones objetivo aceptadas: {len(accepted_ann):,}")
print(f"Anotaciones usadas para ignorar ventanas: {len(ignored_ann):,}")
print(f"Ventanas positivas: {eval_scored['truth'].sum():,}/{len(eval_scored):,}")
print(f"Ventanas excluidas: {evaluation['ignored'].sum():,}")

In [ ]:
# @title Descripción de scores por clase real

display(
    eval_scored.groupby("truth")["score"]
    .describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
    .rename(index={False: "negative", True: "positive"})
)

fig, ax = plt.subplots(figsize=(8, 4.5))
bins = np.linspace(eval_scored["score"].min(), eval_scored["score"].max(), 45)
for truth_value, label, color in [
    (False, "Negativo", "#4C78A8"),
    (True, "Positivo", "#E45756"),
]:
    values = eval_scored.loc[eval_scored["truth"].eq(truth_value), "score"]
    ax.hist(values, bins=bins, alpha=0.55, density=True, label=label, color=color)
ax.axvline(DECISION_THRESHOLD, color="black", linestyle="--", label="Umbral bloqueado")
ax.set(xlabel=SCORE_COLUMN, ylabel="Densidad", title="Distribución de scores por clase real")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# @title Funciones de métricas

def safe_divide(numerator, denominator):
    return float(numerator / denominator) if denominator else np.nan


def binary_metrics(y_true, y_pred, evaluated_seconds=None):
    y_true = np.asarray(y_true, dtype=bool)
    y_pred = np.asarray(y_pred, dtype=bool)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[False, True]).ravel()

    precision = safe_divide(tp, tp + fp)
    recall = safe_divide(tp, tp + fn)
    specificity = safe_divide(tn, tn + fp)
    npv = safe_divide(tn, tn + fn)

    result = {
        "n": int(len(y_true)),
        "positives": int(y_true.sum()),
        "prevalence": float(y_true.mean()) if len(y_true) else np.nan,
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "precision": precision,
        "recall_sensitivity": recall,
        "specificity": specificity,
        "negative_predictive_value": npv,
        "accuracy": safe_divide(tp + tn, tp + tn + fp + fn),
        "balanced_accuracy": np.nanmean([recall, specificity]),
        "F1": safe_divide(2 * precision * recall, precision + recall),
        "false_positive_rate": safe_divide(fp, fp + tn),
        "false_negative_rate": safe_divide(fn, fn + tp),
        "MCC": float(matthews_corrcoef(y_true, y_pred)),
    }
    if evaluated_seconds is not None:
        hours = float(np.sum(evaluated_seconds) / 3600)
        result["evaluated_audio_hours"] = hours
        result["false_positive_windows_per_hour"] = safe_divide(fp, hours)
    return result


def ranking_metrics(y_true, scores):
    y_true = np.asarray(y_true, dtype=bool)
    scores = np.asarray(scores, dtype=float)
    if np.unique(y_true).size < 2:
        return {"ROC_AUC": np.nan, "average_precision": np.nan}
    return {
        "ROC_AUC": float(roc_auc_score(y_true, scores)),
        "average_precision": float(average_precision_score(y_true, scores)),
    }


fixed_metrics = binary_metrics(
    eval_scored["truth"],
    eval_scored["predicted"],
    evaluated_seconds=eval_scored["window_duration"],
)
fixed_metrics.update(ranking_metrics(eval_scored["truth"], eval_scored["score"]))
fixed_metrics["decision_threshold_score"] = float(DECISION_THRESHOLD)
fixed_metrics["decision_threshold_probability"] = threshold_to_probability(
    DECISION_THRESHOLD
)

fixed_metrics_table = pd.Series(fixed_metrics, name="value").to_frame()
display(fixed_metrics_table)

In [ ]:
# @title Matriz de confusión al umbral bloqueado

cm = confusion_matrix(
    eval_scored["truth"], eval_scored["predicted"], labels=[False, True]
)
fig, ax = plt.subplots(figsize=(5.2, 4.6))
image = ax.imshow(cm, cmap="Blues")
for (row, col), value in np.ndenumerate(cm):
    ax.text(col, row, f"{value:,}", ha="center", va="center", fontsize=13)
ax.set_xticks([0, 1], labels=["Predicho −", "Predicho +"])
ax.set_yticks([0, 1], labels=["Real −", "Real +"])
ax.set_title(f"Matriz de confusión — threshold={DECISION_THRESHOLD:g}")
ax.set_xlabel("Predicción")
ax.set_ylabel("Ground truth")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()

In [ ]:
# @title Curvas ROC y Precision–Recall

y_true = eval_scored["truth"].to_numpy(dtype=bool)
scores = eval_scored["score"].to_numpy(dtype=float)

fpr, tpr, roc_thresholds = roc_curve(y_true, scores)
precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_true, scores)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(fpr, tpr, color="#4C78A8", lw=2)
axes[0].plot([0, 1], [0, 1], color="gray", linestyle="--")
axes[0].set(
    xlabel="False positive rate",
    ylabel="True positive rate",
    title=f"ROC — AUC={fixed_metrics['ROC_AUC']:.3f}",
)

axes[1].plot(recall_curve, precision_curve, color="#E45756", lw=2)
axes[1].axhline(y_true.mean(), color="gray", linestyle="--", label="Prevalencia")
axes[1].set(
    xlabel="Recall",
    ylabel="Precision",
    title=f"Precision–Recall — AP={fixed_metrics['average_precision']:.3f}",
    xlim=(0, 1.01),
    ylim=(0, 1.01),
)
axes[1].legend()
fig.tight_layout()
plt.show()

In [ ]:
# @title Barrido de umbrales (diagnóstico exploratorio)

def threshold_performance_table(y_true, scores, window_seconds):
    thresholds = np.unique(np.r_[np.asarray(scores, dtype=float), DECISION_THRESHOLD])
    rows = []
    for threshold in thresholds:
        metrics = binary_metrics(
            y_true,
            np.asarray(scores) >= threshold,
            evaluated_seconds=window_seconds,
        )
        metrics["threshold_score"] = float(threshold)
        metrics["threshold_probability"] = threshold_to_probability(threshold)
        rows.append(metrics)
    return pd.DataFrame(rows).sort_values("threshold_score").reset_index(drop=True)


threshold_results = threshold_performance_table(
    eval_scored["truth"],
    eval_scored["score"],
    eval_scored["window_duration"],
)
best_f1 = (
    threshold_results.sort_values(["F1", "threshold_score"], ascending=[False, False])
    .iloc[0]
    .to_frame("exploratory_best_F1")
)

print("Mejor F1 observado en test (exploratorio; no usar como estimación final imparcial):")
display(
    best_f1.loc[
        [
            "threshold_score",
            "threshold_probability",
            "TP",
            "FP",
            "FN",
            "TN",
            "precision",
            "recall_sensitivity",
            "specificity",
            "F1",
        ]
    ]
)

In [ ]:
# @title Sensibilidad de las métricas al umbral

fig, ax = plt.subplots(figsize=(9, 5))
for metric, color in [
    ("precision", "#4C78A8"),
    ("recall_sensitivity", "#E45756"),
    ("specificity", "#72B7B2"),
    ("F1", "#F2CF5B"),
]:
    ax.plot(threshold_results["threshold_score"], threshold_results[metric], label=metric, color=color)
ax.axvline(DECISION_THRESHOLD, color="black", linestyle="--", label="umbral bloqueado")
finite_scores = eval_scored["score"].quantile([0.01, 0.99]).to_numpy()
ax.set_xlim(*finite_scores)
ax.set_ylim(0, 1.02)
ax.set(xlabel=SCORE_COLUMN, ylabel="Métrica", title="Trade-off según el umbral")
ax.legend(ncol=2)
fig.tight_layout()
plt.show()

In [ ]:
# @title Análisis de sensibilidad a la regla temporal

temporal_rows = []
for rule in ("start", "midpoint", "any_overlap"):
    truth_rule = mark_windows(
        pred,
        accepted_ann,
        rule=rule,
        min_overlap_seconds=MIN_OVERLAP_SECONDS,
    )
    keep = ~evaluation["ignored"]
    row = binary_metrics(
        truth_rule[keep],
        pred.loc[keep, SCORE_COLUMN].to_numpy() >= DECISION_THRESHOLD,
        evaluated_seconds=pred.loc[keep, "window_duration"],
    )
    row.update(ranking_metrics(truth_rule[keep], pred.loc[keep, SCORE_COLUMN]))
    row["truth_assignment"] = rule
    temporal_rows.append(row)

temporal_sensitivity = pd.DataFrame(temporal_rows).set_index("truth_assignment")
display(
    temporal_sensitivity[
        [
            "positives",
            "TP",
            "FP",
            "FN",
            "TN",
            "precision",
            "recall_sensitivity",
            "specificity",
            "F1",
            "ROC_AUC",
            "average_precision",
        ]
    ]
)

In [ ]:
# @title Intervalos de confianza: bootstrap agrupado por archivo

def bootstrap_by_file(data, reps=1_000, seed=42):
    rng = np.random.default_rng(seed)
    files = data["filename"].drop_duplicates().to_numpy()
    groups = {name: group for name, group in data.groupby("filename", sort=False)}
    rows = []

    for _ in range(reps):
        sampled_files = rng.choice(files, size=len(files), replace=True)
        sample = pd.concat([groups[name] for name in sampled_files], ignore_index=True)
        metrics = binary_metrics(
            sample["truth"],
            sample["predicted"],
            evaluated_seconds=sample["window_duration"],
        )
        metrics.update(ranking_metrics(sample["truth"], sample["score"]))
        rows.append(metrics)

    return pd.DataFrame(rows)


bootstrap_results = bootstrap_by_file(
    eval_scored,
    reps=BOOTSTRAP_REPS,
    seed=RANDOM_SEED,
)

ci_metrics = [
    "precision",
    "recall_sensitivity",
    "specificity",
    "accuracy",
    "balanced_accuracy",
    "F1",
    "MCC",
    "ROC_AUC",
    "average_precision",
    "false_positive_windows_per_hour",
]
bootstrap_ci = pd.DataFrame(
    {
        "estimate": [fixed_metrics[name] for name in ci_metrics],
        "ci_2.5%": [bootstrap_results[name].quantile(0.025) for name in ci_metrics],
        "ci_97.5%": [bootstrap_results[name].quantile(0.975) for name in ci_metrics],
    },
    index=ci_metrics,
)
bootstrap_ci.index.name = "metric"
display(bootstrap_ci)

In [ ]:
# @title Evaluación complementaria por archivo y por evento anotado

file_results = (
    eval_scored.groupby("filename", as_index=False)
    .agg(
        truth=("truth", "max"),
        predicted=("predicted", "max"),
        max_score=("score", "max"),
        n_windows=("filename", "size"),
    )
)
file_metrics = binary_metrics(file_results["truth"], file_results["predicted"])
file_metrics.update(ranking_metrics(file_results["truth"], file_results["max_score"]))
file_metrics_table = pd.Series(file_metrics, name="value").to_frame()


def evaluate_annotated_events(events, windows, threshold, tolerance=0.0):
    rows = []
    for _, event in events.iterrows():
        candidates = windows[windows["filename"].eq(event["filename"])]
        if not candidates.empty:
            overlap = overlap_matrix(
                candidates["window_start"],
                candidates["window_end"],
                [event["annotation_start"]],
                [event["annotation_end"]],
                tolerance=tolerance,
            )[0]
            candidates = candidates.loc[overlap > 0]

        row = event.to_dict()
        row["has_model_coverage"] = not candidates.empty
        row["max_overlapping_score"] = (
            float(candidates["score"].max()) if not candidates.empty else np.nan
        )
        row["detected"] = bool(
            not candidates.empty and (candidates["score"] >= threshold).any()
        )
        rows.append(row)
    return pd.DataFrame(rows)


event_results = evaluate_annotated_events(
    accepted_ann,
    evaluation,
    DECISION_THRESHOLD,
    tolerance=EVENT_TOLERANCE_SECONDS,
)
event_evaluable = event_results[event_results["has_model_coverage"]]
event_summary = pd.Series(
    {
        "annotated_events": len(event_results),
        "evaluable_events": len(event_evaluable),
        "detected_events": int(event_evaluable["detected"].sum()),
        "missed_events": int((~event_evaluable["detected"]).sum()),
        "event_recall": float(event_evaluable["detected"].mean()),
        "events_without_model_coverage": int((~event_results["has_model_coverage"]).sum()),
    },
    name="value",
).to_frame()

print("Desempeño por archivo:")
display(file_metrics_table)
print("Sensibilidad por evento anotado:")
display(event_summary)

In [ ]:
# @title Casos para revisión manual y exportación

false_positive_windows = eval_scored[eval_scored["error_type"].eq("FP")].copy()
false_negative_windows = eval_scored[eval_scored["error_type"].eq("FN")].copy()
missed_annotations = event_results[
    event_results["has_model_coverage"] & ~event_results["detected"]
].copy()

display(false_positive_windows.nlargest(10, "score"))
display(false_negative_windows.nsmallest(10, "score"))

if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    fixed_metrics_table.to_csv(OUTPUT_DIR / "metrics_fixed_threshold.csv")
    bootstrap_ci.to_csv(OUTPUT_DIR / "metrics_bootstrap_ci.csv")
    threshold_results.to_csv(OUTPUT_DIR / "threshold_performance.csv", index=False)
    temporal_sensitivity.to_csv(OUTPUT_DIR / "truth_assignment_sensitivity.csv")
    coverage.to_csv(OUTPUT_DIR / "coverage_audit.csv", index=False)
    eval_scored.to_csv(OUTPUT_DIR / "window_predictions_evaluated.csv", index=False)
    false_positive_windows.to_csv(OUTPUT_DIR / "false_positive_windows.csv", index=False)
    false_negative_windows.to_csv(OUTPUT_DIR / "false_negative_windows.csv", index=False)
    file_results.to_csv(OUTPUT_DIR / "file_level_results.csv", index=False)
    event_results.to_csv(OUTPUT_DIR / "event_detection_results.csv", index=False)
    missed_annotations.to_csv(OUTPUT_DIR / "missed_annotations.csv", index=False)

    summary = {
        "prediction_label": PREDICTION_LABEL,
        "target_annotation_labels": list(TARGET_ANNOTATION_LABELS),
        "score_column": SCORE_COLUMN,
        "score_type": SCORE_TYPE,
        "decision_threshold_score": float(DECISION_THRESHOLD),
        "decision_threshold_probability": threshold_to_probability(DECISION_THRESHOLD),
        "truth_assignment": TRUTH_ASSIGNMENT,
        "accepted_qualities": None if ACCEPTED_QUALITIES is None else list(ACCEPTED_QUALITIES),
        "ignored_annotation_labels": list(IGNORED_ANNOTATION_LABELS),
        "bootstrap_reps": int(BOOTSTRAP_REPS),
        "window_level_metrics": {
            key: (None if pd.isna(value) else float(value))
            for key, value in fixed_metrics.items()
        },
        "event_recall": float(event_evaluable["detected"].mean()),
    }
    (OUTPUT_DIR / "evaluation_summary.json").write_text(
        json.dumps(summary, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    print(f"Resultados guardados en: {OUTPUT_DIR.resolve()}")
    for path in sorted(OUTPUT_DIR.iterdir()):
        print(" -", path.name)

## Cómo interpretar el resultado final

- Para documentar el desempeño definitivo, reporte las métricas de
  `metrics_fixed_threshold.csv` y sus intervalos de
  `metrics_bootstrap_ci.csv`.
- Use `threshold_performance.csv` para estudiar el trade-off operativo, pero no
  presente el máximo F1 del propio test como una estimación imparcial.
- Revise `false_positive_windows.csv`, `false_negative_windows.csv` y
  `missed_annotations.csv` en Raven Pro para caracterizar errores sistemáticos.
- Si `PRTR1`, `PRTR2` y `PRTR3` son vocalizaciones de la misma especie y el objetivo
  real es presencia de la especie, repita el análisis incluyendo las tres en
  `TARGET_ANNOTATION_LABELS`. Si el objetivo es detectar exclusivamente PRTR1,
  mantenga la configuración actual y decida explícitamente si PRTR2/PRTR3 deben
  actuar como negativos o como intervalos ignorados.